# Sistema de Recomendación de Productos
## Joyería Diana Laura — Análisis con Algoritmo Apriori

Este notebook implementa un sistema de recomendación basado en reglas de asociación (Apriori).
El objetivo es identificar qué productos se compran frecuentemente juntos para sugerir artículos
relacionados a los clientes durante su proceso de compra.

## 1. Importación de librerías

In [ ]:
import os
import psycopg2
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import matplotlib
import pickle
from mlxtend.frequent_patterns import apriori, association_rules
from mlxtend.preprocessing import TransactionEncoder
from dotenv import load_dotenv

load_dotenv()
print('✅ Librerías cargadas correctamente')

## 2. Conexión a la base de datos

In [ ]:
DATABASE_URL = os.getenv('DATABASE_URL')

conn = psycopg2.connect(DATABASE_URL)
print('✅ Conexión exitosa a Supabase')

## 3. Extracción de datos de ventas

In [ ]:
query = """
    SELECT 
        v.id AS venta_id,
        v.folio,
        c.nombre AS cliente,
        dv.producto_nombre,
        v.fecha_creacion
    FROM ventas v
    JOIN clientes c ON c.id = v.cliente_id
    JOIN detalle_ventas dv ON dv.venta_id = v.id
    ORDER BY v.id
"""

df_raw = pd.read_sql(query, conn)
print(f'✅ Total de registros obtenidos: {len(df_raw)}')
df_raw.head(10)

## 4. Exploración de datos

In [ ]:
print('=== RESUMEN DE DATOS ===')
print(f'Total de ventas (tickets): {df_raw["venta_id"].nunique()}')
print(f'Total de clientes: {df_raw["cliente"].nunique()}')
print(f'Total de productos distintos: {df_raw["producto_nombre"].nunique()}')
print()
print('=== CLIENTES ===')
print(df_raw['cliente'].value_counts())
print()
print('=== PRODUCTOS MÁS FRECUENTES ===')
print(df_raw['producto_nombre'].value_counts())

In [ ]:
# Productos más vendidos
fig, ax = plt.subplots(figsize=(12, 5))
conteo = df_raw['producto_nombre'].value_counts()
conteo.plot(kind='bar', ax=ax, color='#ECB2C3', edgecolor='#c9a84c')
ax.set_title('Frecuencia de productos en tickets de venta', fontsize=14, fontweight='bold')
ax.set_xlabel('Producto')
ax.set_ylabel('Veces que aparece en ventas')
plt.xticks(rotation=45, ha='right', fontsize=8)
plt.tight_layout()
plt.show()

In [ ]:
# Ventas por cliente
ventas_por_cliente = df_raw.groupby('cliente')['venta_id'].nunique()
fig, ax = plt.subplots(figsize=(8, 4))
ventas_por_cliente.plot(kind='bar', ax=ax, color='#c9a84c', edgecolor='#ECB2C3')
ax.set_title('Número de compras por cliente', fontsize=13, fontweight='bold')
ax.set_xlabel('Cliente')
ax.set_ylabel('Número de compras')
plt.xticks(rotation=30, ha='right', fontsize=9)
plt.tight_layout()
plt.show()

## 5. Preparación de tickets para el modelo

In [ ]:
# Agrupar por venta_id para obtener los tickets
tickets = df_raw.groupby('venta_id')['producto_nombre'].apply(list).tolist()

print(f'Total de tickets: {len(tickets)}')
print()
print('Ejemplo de tickets:')
for i, t in enumerate(tickets[:5]):
    print(f'  Ticket {i+1}: {t}')

In [ ]:
# Codificación de transacciones
te = TransactionEncoder()
te_array = te.fit(tickets).transform(tickets)
df_encoded = pd.DataFrame(te_array, columns=te.columns_)

print(f'Dimensiones de la matriz: {df_encoded.shape}')
print(f'Productos (columnas): {list(df_encoded.columns)}')
df_encoded.head()

## 6. Entrenamiento del modelo Apriori

In [ ]:
# Encontrar itemsets frecuentes
frequent_itemsets = apriori(df_encoded, min_support=0.1, use_colnames=True)
frequent_itemsets['longitud'] = frequent_itemsets['itemsets'].apply(len)

print(f'Total de itemsets frecuentes encontrados: {len(frequent_itemsets)}')
frequent_itemsets.sort_values('support', ascending=False)

In [ ]:
# Generar reglas de asociación
rules = association_rules(frequent_itemsets, metric='lift', min_threshold=1.1, num_itemsets=len(frequent_itemsets))
rules = rules.sort_values('confidence', ascending=False)

print(f'Total de reglas generadas: {len(rules)}')
rules[['antecedents', 'consequents', 'support', 'confidence', 'lift']]

## 7. Visualización de reglas

In [ ]:
# Gráfica de soporte vs confianza
fig, ax = plt.subplots(figsize=(8, 5))
scatter = ax.scatter(
    rules['support'],
    rules['confidence'],
    c=rules['lift'],
    cmap='RdPu',
    s=200,
    edgecolors='#c9a84c',
    linewidths=1.5
)
plt.colorbar(scatter, label='Lift')
ax.set_xlabel('Soporte', fontsize=11)
ax.set_ylabel('Confianza', fontsize=11)
ax.set_title('Reglas de Asociación — Soporte vs Confianza', fontsize=13, fontweight='bold')
plt.tight_layout()
plt.show()

In [ ]:
# Mostrar reglas en formato legible
print('=== REGLAS DE ASOCIACIÓN ENCONTRADAS ===')
print()
for _, row in rules.iterrows():
    antecedente = list(row['antecedents'])[0]
    consecuente = list(row['consequents'])[0]
    print(f'Si compra: "{antecedente}"')
    print(f'  → También suele llevar: "{consecuente}"')
    print(f'     Confianza: {round(row["confidence"]*100, 1)}%  |  Lift: {round(row["lift"], 2)}')
    print()

## 8. Prueba del sistema de recomendación

In [ ]:
def recomendar(productos_carrito, rules, top_n=3):
    recomendaciones = {}
    for producto in productos_carrito:
        filtro = rules[rules['antecedents'].apply(lambda x: producto in x)]
        for _, row in filtro.iterrows():
            for rec in row['consequents']:
                if rec not in productos_carrito:
                    if rec not in recomendaciones:
                        recomendaciones[rec] = 0
                    recomendaciones[rec] += row['confidence']
    
    ordenadas = sorted(recomendaciones.items(), key=lambda x: x[1], reverse=True)
    return [r[0] for r in ordenadas[:top_n]]

# Prueba 1
carrito = ['Aretes corazón doble plata']
recs = recomendar(carrito, rules)
print(f'Carrito: {carrito}')
print(f'Recomendaciones: {recs}')
print()

# Prueba 2
carrito2 = ['Pulsera personalizada Día del Niño']
recs2 = recomendar(carrito2, rules)
print(f'Carrito: {carrito2}')
print(f'Recomendaciones: {recs2}')

## 9. Guardar el modelo

In [ ]:
modelo = {
    'rules': rules,
    'te': te,
    'frequent_itemsets': frequent_itemsets
}

with open('modelo_recomendacion.pkl', 'wb') as f:
    pickle.dump(modelo, f)

print('✅ Modelo guardado como modelo_recomendacion.pkl')

## 10. Conclusiones

El sistema de recomendación basado en el algoritmo Apriori fue implementado exitosamente para Joyería Diana Laura.

**Resultados obtenidos:**
- Se analizaron los tickets de venta históricos de la plataforma
- El modelo identificó patrones de compra entre los productos del catálogo
- Se generaron reglas de asociación con métricas de soporte, confianza y lift

**Interpretación de métricas:**
- **Soporte:** frecuencia con la que aparece la combinación en el total de tickets
- **Confianza:** probabilidad de que al comprar el producto A también se compre el B
- **Lift > 1:** indica que la relación entre productos no es aleatoria, existe una asociación real

**Siguiente paso:** El modelo fue exportado como `modelo_recomendacion.pkl` y será consumido por el microservicio Flask para integrarse con la plataforma web de Joyería Diana Laura.